In [ ]:
# ==============================
# Mask Cancellation Proof
#
# Purpose: prove that the recovery mechanism (rebuilding the ring over
# ONLY active clients, i.e. CANCEL) restores exact mask cancellation
# after dropout, while NOT rebuilding the ring (NO_CANCEL) leaves a
# nonzero residual.
#
# This does NOT require any model training -- it operates purely on
# the masking math itself, using the same MASK_SCALE and the same
# CSV-driven dropout schedule as the full training scripts. This makes
# it a fast, deterministic, seed-robust proof: no training noise,
# no accuracy confound, just the masking arithmetic.
#
# Uses the ACTUAL model parameter count (EMNIST_CNN) as the mask
# vector dimension, so the result matches the real training setup
# exactly rather than an arbitrary placeholder size.
#
# For each round:
#   - selected_clients = 30 clients from the CSV
#   - dropped_clients   = clients to drop, from the CSV
#   - active_clients     = selected - dropped
#
#   CANCEL:    ring built over active_clients only -> masks sum to ~0
#   NO_CANCEL: ring built over selected_clients (all 30), then
#              dropped clients' masks are discarded at aggregation
#              -> masks do NOT sum to 0
#
# What this proves:
#     Recovery:    sum(masks) = 0  (exact cancellation)
#     No recovery: sum(masks) != 0 (nonzero residual)
#
# What this does NOT prove:
#   - It does not claim recovery improves accuracy or learning.
#   - It does not, by itself, demonstrate a successful privacy attack;
#     it shows that cancellation fails without recovery, which is the
#     precondition the masking protocol's security argument relies on.
#
# Output: a per-round table + summary stats of the residual mask norm
# for both variants, saved to mask_cancellation_proof.csv.
# ==============================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import ast

# -------------------------
# Configuration (same constants as the training scripts)
# -------------------------
MASK_SCALE = 0.001
SEED = 42

torch.manual_seed(SEED)

# -------------------------
# Model: CNN (LeNet++) -- same architecture as the training scripts,
# used here only to derive the real parameter count.
# -------------------------
class EMNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64*7*7, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

tmp_model = EMNIST_CNN()
VECTOR_DIM = sum(p.numel() for p in tmp_model.parameters())
print(f"Using actual model parameter count as mask vector dimension: {VECTOR_DIM}")

rng = np.random.default_rng(SEED)

# -------------------------
# Load CSV dropout schedule
# -------------------------
schedule_df = pd.read_csv("client_selection_60_rounds.csv")
schedule_df = schedule_df.sort_values("Round").reset_index(drop=True)

results = []

for _, row in schedule_df.iterrows():
    rnd = int(row["Round"])
    selected_clients = ast.literal_eval(row["Selected_30_Clients"])
    dropped_clients = set(ast.literal_eval(row["Chosen_Clients"]))
    active_clients = [c for c in selected_clients if c not in dropped_clients]

    dummy_vec = np.zeros(VECTOR_DIM)

    # ------------------------
    # CANCEL: ring built over active_clients only
    # ------------------------
    masks_cancel = {cid: np.zeros_like(dummy_vec) for cid in active_clients}
    sorted_active = sorted(active_clients)
    K = len(sorted_active)
    if K > 0:
        for idx, cid in enumerate(sorted_active):
            right = sorted_active[(idx + 1) % K]
            if cid < right:
                rmask = rng.standard_normal(VECTOR_DIM) * MASK_SCALE
                masks_cancel[cid] += rmask
                masks_cancel[right] -= rmask
    cancel_sum = sum(masks_cancel.values(), np.zeros_like(dummy_vec)) if K > 0 else np.zeros_like(dummy_vec)
    cancel_residual_norm = np.linalg.norm(cancel_sum).item()

    # ------------------------
    # NO_CANCEL: ring built over all 30 selected_clients,
    # dropped clients' mask entries are discarded at aggregation
    # ------------------------
    masks_nocancel = {cid: np.zeros_like(dummy_vec) for cid in selected_clients}
    sorted_selected = sorted(selected_clients)
    K2 = len(sorted_selected)
    for idx, cid in enumerate(sorted_selected):
        right = sorted_selected[(idx + 1) % K2]
        if cid < right:
            rmask = rng.standard_normal(VECTOR_DIM) * MASK_SCALE
            masks_nocancel[cid] += rmask
            masks_nocancel[right] -= rmask
    # Only sum the masks of clients that actually get aggregated (active ones)
    nocancel_sum = sum((masks_nocancel[cid] for cid in active_clients), np.zeros_like(dummy_vec))
    nocancel_residual_norm = np.linalg.norm(nocancel_sum).item()

    results.append({
        "Round": rnd,
        "Selected": len(selected_clients),
        "Dropped": len(dropped_clients),
        "Active": len(active_clients),
        "CANCEL_residual_norm": cancel_residual_norm,
        "NO_CANCEL_residual_norm": nocancel_residual_norm,
    })

results_df = pd.DataFrame(results)
results_df.to_csv("mask_cancellation_proof.csv", index=False)

# -------------------------
# Summary
# -------------------------
print("=" * 70)
print("MASK CANCELLATION PROOF -- SUMMARY")
print("=" * 70)
print(results_df.to_string(index=False))
print()
print("-" * 70)
print(f"CANCEL    : mean residual norm = {results_df['CANCEL_residual_norm'].mean():.2e} "
      f"(max = {results_df['CANCEL_residual_norm'].max():.2e})")
print(f"NO_CANCEL : mean residual norm = {results_df['NO_CANCEL_residual_norm'].mean():.6f} "
      f"(max = {results_df['NO_CANCEL_residual_norm'].max():.6f})")
print("-" * 70)

rounds_with_dropout = results_df[results_df["Dropped"] > 0]
print(f"\nRounds with dropout (k>0): {len(rounds_with_dropout)} / {len(results_df)}")
print(f"  CANCEL residual on these rounds    : mean={rounds_with_dropout['CANCEL_residual_norm'].mean():.2e}")
print(f"  NO_CANCEL residual on these rounds  : mean={rounds_with_dropout['NO_CANCEL_residual_norm'].mean():.6f}")

rounds_no_dropout = results_df[results_df["Dropped"] == 0]
print(f"\nRounds with NO dropout (k=0): {len(rounds_no_dropout)} / {len(results_df)}")
print(f"  CANCEL residual    : mean={rounds_no_dropout['CANCEL_residual_norm'].mean():.2e}")
print(f"  NO_CANCEL residual : mean={rounds_no_dropout['NO_CANCEL_residual_norm'].mean():.2e}  "
      f"(matches CANCEL -- with no dropout, the ring is identical in both methods)")

print("\nConclusion:")
print("  CANCEL keeps the residual mask norm approximately zero")
print("  (floating-point precision only) in every round, regardless")
print("  of dropout severity.")
print("  NO_CANCEL's residual norm grows with the number of dropped")
print("  clients, showing that without rebuilding the ring (recovery),")
print("  the masking scheme fails to achieve exact cancellation and")
print("  introduces a nonzero, dropout-dependent residual into the")
print("  aggregate.")
print()
print("  This demonstrates: recovery -> exact mask cancellation;")
print("  no recovery -> residual mask remains. It does not, by itself,")
print("  claim anything about accuracy or learning performance.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Smaller fonts (fix clutter) ----------
plt.rcParams.update({
    "font.size": 18,
    "axes.labelsize": 19,
    "axes.titlesize": 19,
    "legend.fontsize": 17,
    "xtick.labelsize": 17,
    "ytick.labelsize": 17,
    "lines.linewidth": 2
})

df = pd.read_csv("mask_cancellation_proof.csv")

# ---------- BIGGER figure (main fix) ----------
fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(9, 7),   # ⬅️ increased size
    sharex=True,
    gridspec_kw={"height_ratios": [2, 1], "hspace": 0.10}
)

# ---------------- Panel A ----------------
ax1.plot(df["Round"], df["NO_CANCEL_residual_norm"],
         color="#d62728",
         label="No Recovery")

ax1.plot(df["Round"], df["CANCEL_residual_norm"],
         color="#1f77b4",
         label="Recovery")

ax1.set_yscale("log")
ax1.set_ylabel(r"Residual Mask Norm")

ax1.grid(True, linestyle="--", alpha=0.4, linewidth=0.5)

ax1.legend(loc="center right", frameon=True)

# reference floor
ax1.axhline(1e-15, color="black", linestyle=":", linewidth=0.8)

# ---------------- Panel B ----------------
ax2.bar(df["Round"], df["Dropped"],
        color="#7f8c8d",
        width=0.8)

ax2.set_ylabel("Clients Dropped")
ax2.set_xlabel("Round")
ax2.grid(True, axis="y", linestyle="--", alpha=0.4, linewidth=0.5)

plt.tight_layout()

plt.savefig("mask_cancellation_figure.png", dpi=300, bbox_inches="tight")

from google.colab import files
files.download("mask_cancellation_figure.png")

print("Saved and downloaded figure")

print("Saved figure")